# Lab 02: From notebook to a command you can run again

Automated Workflows in Satellite Image Processing · v1.0 · 5 October 2026

Your Lab 01 service discovers real Sentinel scenes and stores its progress. Today you turn it into a program you can run from a terminal whenever your laptop is on.

Use this notebook to predict behavior, run checks and explain results. Edit **student_tasks.py** in a text editor. Run the actual service from a terminal. The service never depends on a notebook kernel.

By the end you can run the same command several times over a week, inspect its history, and explain whether it succeeded even when it found zero new scenes.

Suggested lab time: 90 minutes, followed by four short runs over seven days. No GPU, cluster or continuously running laptop is needed.


## What comes from Lab 01?

`lab01_service.py` contains the working schema, database insertion, pagination, overlapping discovery, and safe one-JPEG downloader extracted from Lab 01 instructor v3. It has no notebook execution side effects.

`workflow.py` supplies the orchestration and two small additions: a run journal and a link between each query stream and its scenes. The link prevents selecting a preview from a different region when a database contains several streams.

`sentinel_cli.py` supplies the argument parser. Your three repairs go in `student_tasks.py`. The captured provider pages and real preview in `fixtures/` are reused from Lab 01 only for repeatable checks. Operational discovery always uses live HTTPS requests; it never substitutes recorded data after a network failure.

Do not edit the supplied service or add a timer loop.


## Setup

Extract the student ZIP. Open a terminal in the extracted folder containing `sentinel_cli.py`, using your existing Python environment. If you need the notebook interface:

```bash
python -m pip install -r requirements.txt
```

```bash
python -m jupyterlab Lab02_Sentinel_CLI_v1_20261005.ipynb
```

The command line service and checks themselves need no third party packages. Keep the folder together. If you launch Jupyter from somewhere else, set `PROJECT` below to this folder's absolute path.


In [1]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

PROJECT = Path.cwd().resolve()
assert (PROJECT / 'sentinel_cli.py').is_file(), 'Set PROJECT to the extracted student folder'

def run_check(exercise=None):
    args = [sys.executable, str(PROJECT / 'check_lab02.py')]
    if exercise is not None:
        args += ['--exercise', str(exercise)]
    result = subprocess.run(args, cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('Exit code:', result.returncode)
    return result.returncode

print('Project:', PROJECT)


Project: /Users/molczane/Developer/Automated_Workflows/Labs/Lab02


## Before editing: inspect the command interface (5 minutes)

Run these in your terminal:

```bash
python sentinel_cli.py --help
```

```bash
python sentinel_cli.py discover --help
```

Read `config/krakow_week.json`. Where is the region specified? What makes this a repeatable query? Why is there an initial start but no permanent end date?

There are deliberate bugs in the three task functions. The historical demonstration end means the starter is not ready for the weekly experiment.

### Completed explanation

The region is the EPSG:4326 `bbox` in `config/krakow_week.json`:
[19.7, 49.9, 20.1, 50.2], covering Kraków. Endpoint, collection and bounding box
identify the query stream. `initial_start` is fixed at 21 September 2026, whereas
the end is captured anew for each invocation; this lets later runs discover newer
acquisitions without discarding saved progress.

Before repair, the three exercise checks failed on configuration-relative paths,
the fixed historical end and the false success exit code. The supplied recovery
and real-preview checks already passed. The supplied CLI, workflow, Lab 01 service
and all checks are unchanged.


## Exercise 1: One configuration, one persistent state (20 minutes)

Read `resolve_paths` in `student_tasks.py`. Imagine running the program first from its own folder and then from its parent. Predict which database each invocation opens.

Repair the function so relative database, preview and log paths are anchored to the configuration file's parent. Keep absolute paths absolute. Resolve paths without creating files or changing the process working directory.

This matters because the scheduler or terminal decides the working directory. A successful command can silently create a second, empty catalogue if path handling is wrong.

Run the check before and after the repair:


In [2]:
run_check(1)


PASS: 1

Exit code: 0


0

Write your explanation here: why is anchoring these paths to the configuration file more reliable than anchoring them to the terminal folder? What would happen if you deliberately changed the database path?

### Completed explanation

Before repair, `../state/catalogue.sqlite` was resolved against the terminal's
working directory. Launching from Lab02 would therefore point outside Lab02,
while launching from its parent would select another database again. This could
make a restarted service appear to lose its history.

`resolve_paths` now uses the resolved configuration file's parent for all relative
database, preview and log paths. It returns absolute Path objects, leaves an
already absolute path unchanged, and neither creates files nor changes the
working directory. In particular, preserving an absolute path also preserves its
spelling when the OS exposes a directory through a symlink.

Changing the configured database path deliberately selects another catalogue.
If that destination is new, discovery starts a separate history at initial_start;
it does not automatically migrate the previous database or previews. Keep this
path unchanged throughout the weekly experiment.


## Exercise 2: Discover through the current time (20 minutes)

Read `choose_until`. The prototype copied Lab 01's demonstration end. That was useful for a fixed experiment, but it cannot discover this week's acquisitions.

Repair it so `--until` takes priority when supplied; otherwise capture an aware UTC timestamp once per invocation. Normalize explicit timestamps to UTC and reject values without a timezone. The optional `clock` argument lets a check control time without changing the satellite records.

Do not set the search start to “now minus one day.” The supplied discovery service uses the saved watermark minus the overlap, bounded by the original initial start.


In [3]:
run_check(2)


PASS: 2

Exit code: 0


0

Write your explanation here: if you run on Monday and next on Thursday, which part of the configuration and saved state determines the next search interval? Why must the upper bound remain fixed while following pages?

### Completed explanation

On Thursday the lower bound is the later of `initial_start` and Monday's saved
watermark minus `overlap_days` (seven days in this configuration). It is not
Thursday minus one day: a laptop that skipped days resumes from its last completed
search, with overlap for late publication.

An explicit `--until` takes priority, is parsed and normalized to UTC, and never
consults the clock. Without it, `choose_until` captures an aware UTC datetime once;
a naive injected clock value is rejected too. The resulting end stays fixed
across every page, so a long search cannot keep changing its own interval. Only a
complete search advances the saved watermark. A later invocation captures a new
end and repeats the overlap.


## Exercise 3: Make the terminal tell the truth (20 minutes)

Read `execute_job`. Predict what a caller sees when the network request fails. Repair the function so success returns exit code 0 and the actual result; failure logs a useful error and returns exit code 1 with `status='failed'` and an error description.

A completed search with zero new scenes is a successful run. A connection error is a failed run. Do not infer success from a positive scene count.

The check also launches the actual CLI with a missing configuration file to verify that the exit code reaches the terminal.


In [4]:
run_check(3)


PASS: 3

Exit code: 0


0

After your repair, try this deliberate failure in the terminal:

```bash
python sentinel_cli.py discover --config config/missing.json
```

Inspect the exit code immediately afterward. In Git Bash or Bash:

```bash
echo $?
```

In PowerShell:

```powershell
$LASTEXITCODE
```

Explain what a future scheduler could do with this code. A missing configuration cannot provide a log path, so this early error appears on the terminal; operational errors also go to the configured log.

### Completed explanation

Before repair, a connection error was swallowed and reported as exit 0 with
`status='succeeded'`. `execute_job` now returns the actual result with exit 0 on
success, or logs the exception with its traceback and returns exit 1 with
`status='failed'` plus the exception type and message on failure. Zero newly
associated scenes remains a valid success.

The actual missing-configuration command returned exit 1 and a JSON
FileNotFoundError diagnostic. A future scheduler can use that exit code to flag a
failed run and apply its retry policy; it should not infer failure from a zero
scene count. Because the missing configuration cannot specify a log file, this
error was printed to stderr; configured operational failures are also logged to
the configured events.log. Argparse syntax errors remain the supplied CLI's exit 2.


## Exercise 4: Verify repeated execution (15 minutes)

Run every check. In addition to your three repairs, these reuse actual captured Lab 01 records to verify interrupted discovery, recovery, repeat sightings, and one verified preview. Only clocks and transport failures are controlled.


In [5]:
run_check()


PASS: 1
PASS: 2
PASS: 3
PASS: service recovery
PASS: one real preview

Exit code: 0


0

Now run the live program. Run one command at a time; simultaneous writers are outside this lab's scope. The first command discovers metadata and requests at most one preview. Preview download is optional on subsequent runs.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

```bash
python sentinel_cli.py discover --config config/krakow_week.json
```

```bash
python sentinel_cli.py report --config config/krakow_week.json
```

To demonstrate stable paths, move to the parent of the extracted folder and invoke the same program/configuration with their paths. Adapt the folder name to your extraction:

```bash
python Lab02_Sentinel_CLI_v1_20261005/sentinel_cli.py report --config Lab02_Sentinel_CLI_v1_20261005/config/krakow_week.json
```

Compare the reported absolute database path with the earlier report. It must be identical.

Live availability is not guaranteed. If a request fails, keep the log, inspect the saved watermark and retry later. Do not treat an offline check as evidence of a successful live run.

### Completed explanation

The terminal executed the initial live command and its immediate rerun against
the real Earth Search endpoint. The first returned 3 pages,
40 sightings and 40 newly associated scenes;
the rerun returned 2 pages, 8 sightings and
0 new scenes. The first run saved one actual JPEG for
`S2C_34UCA_20261025_0_L2A` (12,323 bytes). The second run was
metadata-only, as permitted by the exercise.

Reports launched from Lab02 and from Labs returned the same absolute database
path and identical run histories. Both resolved to `Lab02/state/catalogue.sqlite`.
A wider `--lookback-days 30` reconciliation also completed successfully, with
36 sightings and 0 new scenes.
The configuration, initial_start and database were kept unchanged. Recorded
terminal results are shown below; the notebook does not launch another discovery
when it is executed.


In [6]:
# Actual terminal evidence from the day-one experiment (read only).
# Saved outputs remain visible even when the local ignored state is not submitted.
for label in ('missing-config', 'first-live', 'immediate-rerun',
              'report-local', 'report-parent', 'reconciliation'):
    evidence_file = PROJECT / 'state' / 'evidence' / f'{label}.json'
    if not evidence_file.is_file():
        print(label, ': local evidence absent; consult saved notebook output.')
        continue
    evidence = json.loads(evidence_file.read_text())
    print('Command:', ' '.join(evidence['command'][1:]))
    print('Working directory:', evidence['cwd'])
    print('Exit code:', evidence['exit_code'])
    print(evidence['stdout'])


Command: sentinel_cli.py discover --config config/missing.json
Working directory: /Users/molczane/Developer/Automated_Workflows/Labs/Lab02
Exit code: 1
{
  "status": "failed",
  "error": "FileNotFoundError: [Errno 2] No such file or directory: 'config/missing.json'"
}

Command: sentinel_cli.py discover --config config/krakow_week.json --preview
Working directory: /Users/molczane/Developer/Automated_Workflows/Labs/Lab02
Exit code: 0
{
  "run_id": "54ec0dc4-ffe9-45cc-9065-68ba8ee08878",
  "stream_key": "9ae6b6299dd613ee619b8bb36c584f70051bb153fd582d37bc644c3ca29a0420",
  "started_at": "2026-10-25T16:47:32.918946Z",
  "finished_at": "2026-10-25T16:47:36.349573Z",
  "status": "succeeded",
  "search_start": "2026-09-21T00:00:00Z",
  "search_until": "2026-10-25T16:47:32.917116Z",
  "pages": 3,
  "sightings": 40,
  "new_scenes": 40,
  "latest_acquired": "2026-10-25T09:46:46.101000Z",
  "preview_item": "S2C_34UCA_20261025_0_L2A",
  "preview_path": "/Users/molczane/Developer/Automated_Workflows

## Inspect your actual state

This cell only reads the current project's database. It does not perform discovery. If you changed the configured database path, update `DB` accordingly.


In [7]:
DB = (PROJECT / 'state' / 'catalogue.sqlite').resolve()
if not DB.is_file():
    print('Run live discovery from the terminal first; no database is created here.')
else:
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        conn.row_factory = sqlite3.Row
        runs = [dict(row) for row in conn.execute(
            'SELECT started_at, status, search_start, search_until, pages, sightings, new_scenes, latest_acquired, error FROM runs ORDER BY started_at, rowid')]
        print(json.dumps(runs, indent=2))
        print('Saved query watermarks:')
        for row in conn.execute('SELECT stream_key, watermark FROM streams'):
            print(dict(row))


[
  {
    "started_at": "2026-10-25T16:47:32.918946Z",
    "status": "succeeded",
    "search_start": "2026-09-21T00:00:00Z",
    "search_until": "2026-10-25T16:47:32.917116Z",
    "pages": 3,
    "sightings": 40,
    "new_scenes": 40,
    "latest_acquired": "2026-10-25T09:46:46.101000Z",
    "error": null
  },
  {
    "started_at": "2026-10-25T16:47:36.385439Z",
    "status": "succeeded",
    "search_start": "2026-10-18T16:47:32.917116Z",
    "search_until": "2026-10-25T16:47:36.384964Z",
    "pages": 2,
    "sightings": 8,
    "new_scenes": 0,
    "latest_acquired": "2026-10-25T09:46:46.101000Z",
    "error": null
  },
  {
    "started_at": "2026-10-25T16:47:38.122303Z",
    "status": "succeeded",
    "search_start": "2026-09-25T16:47:36.384964Z",
    "search_until": "2026-10-25T16:47:38.121854Z",
    "pages": 3,
    "sightings": 36,
    "new_scenes": 0,
    "latest_acquired": "2026-10-25T09:46:46.101000Z",
    "error": null
  }
]
Saved query watermarks:
{'stream_key': '9ae6b6299dd61

Explain your two consecutive runs. Which number counts sightings, and which counts scene identities first associated with this stream? Why may the second run have many sightings and zero new scenes? If new imagery arrived between runs, explain the difference rather than requiring a zero count.

If discovery fails after page 1, committed scenes remain and the discovery watermark does not advance. If discovery completes but its optional preview fails, the command fails while the discovery watermark has already advanced. These outcomes are recorded separately in the run journal and download status. Retry the same preview command to recover.

### Completed explanation

The first run saw 40 catalogue records and associated
40 distinct scene identities with this stream for the first
time. The immediate rerun searched from 2026-10-18T16:47:32.917116Z, saw
8 records in the overlapping interval, and added
0 new scene identities. This is expected idempotent behaviour:
seeing an existing scene again does not create a new scene or stream membership.
Both commands completed with exit 0 and succeeded journal rows.

Sightings count records seen in a run, including repeat observations. New scenes
count identities first associated with this stream, not the number of acquisitions
made since the previous command. The supplied recovery check separately confirms
that a page-two failure preserves committed pages and the old watermark, while a
preview failure after successful discovery preserves the advanced watermark and
records the command failure. Those controlled checks are distinct from these
successful live runs.


## The seven day experiment

Run discovery at least four times over seven days whenever your laptop is on and online. Keep the same region, initial start and database throughout. Include one immediate rerun and one gap of at least 48 hours. A short run needs no permanent background process.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

You may omit `--preview` for metadata-only runs. It always selects the newest available suitable JPEG in this stream, without a cloud filter; a very cloudy preview is still a valid acquisition. It may reuse the same verified JPEG on several days. A JPEG is not a scientific raster product or a quality assessment.

At the end, save a report:

```bash
python sentinel_cli.py report --config config/krakow_week.json > week_report.json
```

Complete this table from the actual run history:

| Run time (UTC) | Outcome | Search interval | Sightings | Newly discovered scenes | Newest acquisition | Observation |
|---|---|---|---|---|---|---|
| 2026-10-25T16:47:32.918946Z | succeeded | 2026-09-21T00:00:00Z → 2026-10-25T16:47:32.917116Z | 40 | 40 | 2026-10-25T09:46:46.101000Z | Initial live search; one newest suitable JPEG requested. |
| 2026-10-25T16:47:36.385439Z | succeeded | 2026-10-18T16:47:32.917116Z → 2026-10-25T16:47:36.384964Z | 8 | 0 | 2026-10-25T09:46:46.101000Z | Immediate rerun; overlap revisited existing scene identities. |
| 2026-10-25T16:47:38.122303Z | succeeded | 2026-09-25T16:47:36.384964Z → 2026-10-25T16:47:38.121854Z | 36 | 0 | 2026-10-25T09:46:46.101000Z | 30-day reconciliation; no new scene identities found. |

Discuss: did skipping days lose scenes within the covered search interval? How old was the newest acquisition? Did a new discovery necessarily mean a recent acquisition? What evidence distinguishes “no new scenes” from “the request failed”? Was the newest preview also the clearest?

This is polling, not an immediate notification service. Acquisition time is not catalogue publication time. Finite overlap can miss items published very late with older acquisition dates; a complete search also cannot prove that the provider has every acquisition. Try a wider lookback as a reconciliation exercise:

```bash
python sentinel_cli.py discover --config config/krakow_week.json --lookback-days 30
```

This cannot go before `initial_start`. Set the initial start once before your first operational run; do not move it forward every day. The shipped start is 21 September 2026, two weeks before this edition. For a future teaching date, update it before starting the experiment.

### Completed explanation

**Experiment in progress.** These are three actual invocations on 25 October
2026, not evidence of a completed seven-day experiment. The immediate-rerun
requirement is met. There is not yet a 48-hour gap or a seven-day span; no run times
have been fabricated or advanced using an injected clock.

To finish, leave the same configuration and local database in place. Perform the
next manual discovery no earlier than **2026-10-27T16:47:40.044439Z** to establish a 48-hour
gap since the last run. Perform a final manual discovery no earlier than
**2026-11-01T16:47:32.918946Z**, seven days after the first run. This produces at least five
actual runs across the required period. If you run in between, use the actual
journal to confirm that some consecutive pair still has a gap of at least 48 hours.

Then regenerate `week_report.json`, update this table and `week_observations.md`
from the actual journal, and rerun the notebook to refresh the displayed history.
The JSON report currently contains the real day-one snapshot and is not a final
weekly report. The separate observation file contains the follow-up commands.

At this snapshot, the newest acquisition is 2026-10-25T09:46:46.101000Z, roughly
7.01 hours before the report. New discovery need not mean
recent acquisition: the first search also found older acquisitions back to the
fixed September start. A successful run with zero new scenes is evidenced by exit
0, a succeeded journal entry and completed watermark; a failed request is evidenced
by exit 1, a failed journal entry and an error. The wider reconciliation found no
additional scenes today, which does not prove the provider's catalogue is complete.

The selected newest preview's whole-tile cloud cover was 99.7338%,
whereas the lowest catalogue cloud cover was 0.004261% for
`S2B_34UDA_20261023_0_L2A`. Selection uses acquisition time rather than minimum cloud
cover. Whole-tile cloud cover cannot certify clarity within our box or the
scientific quality of a JPEG. The question about skipped days remains open until
the actual multi-day observations exist; the expected interval includes missed
days, but finite overlap cannot guarantee recovery of arbitrarily late publication.


## Submission

Submit your repaired `student_tasks.py`, this notebook with checks and explanations, `week_report.json`, and your completed observation table. Include a short relevant log excerpt if a live run failed. Do not commit preview images, database files or the entire log to Git.

Your service is now ready to be run by another program. Containers and cluster scheduling can be added in later labs without moving the operational logic back into the notebook.

Provider documentation: [Element 84 Earth Search](https://github.com/Element84/earth-search). Lab 01 source: `Lab01_Sentinel_RealData_Repair_v3_20261005_Instructor.ipynb`. Recorded fixtures retain the original provider records and real JPEG.

### Completed explanation

The repaired student_tasks.py, executed checks and explanations, real current
week_report.json and day-one observation table are present. The seven-day
experiment remains open for the explicitly listed follow-up runs. Preview images,
the local database, raw terminal evidence and the complete log are under ignored
state/. The deliberate missing-configuration test is expected failure evidence;
all three live discovery commands completed successfully.
